# Simulation Manuscript Figures

Generate publication-quality figures from simulation pipeline outputs.
Adapted from the original `simulation_manuscript_figures.ipynb`.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import warnings
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import seaborn as sns
import scipy.stats

from notebooks._common import load_config

In [ ]:
cfg = load_config(config_path, profile)
sim = cfg["simulation"]
output_dir = sim["output_dir"]
lasso_choice = sim["lasso_choice"]

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
%matplotlib inline

rc_kwargs = {"legend.frameon": False, "font.size": 11, "font.weight": "normal"}
plt.rcParams.update(**rc_kwargs)

## Load pipeline outputs

In [ ]:
simulated_muteffects_df = pd.read_csv(f"{output_dir}/simulated_muteffects.csv")
simulated_func_scores_df = pd.read_csv(f"{output_dir}/simulated_func_scores.csv")
model_vs_truth_df = pd.read_csv(f"{output_dir}/model_vs_truth_beta_shift.csv")
sparsity_df = pd.read_csv(f"{output_dir}/fit_sparsity.csv")
replicate_corr_df = pd.read_csv(f"{output_dir}/library_replicate_correlation.csv")
cv_loss_df = pd.read_csv(f"{output_dir}/cross_validation_loss.csv")

print(f"Loaded simulation outputs from {output_dir}")

## Simulated shift heatmap

In [ ]:
aas = ["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N",
       "P", "Q", "R", "S", "T", "V", "W", "Y", "*"]

data = simulated_muteffects_df.pivot_table(index="mut_aa", columns="site", values="shift")
data = data.clip(lower=-10)

plt.figure(figsize=[12, 5])
g = sns.heatmap(
    data.reindex(aas),
    cmap="RdBu_r",
    center=0,
    xticklabels=True,
    yticklabels=True,
)
g.set_title("Simulated shifts by site and amino acid")
plt.tight_layout()
plt.show()

## Model vs truth: beta and shift correlation

In [ ]:
bottlenecks = {
    "observed_phenotype": "no noise",
    "loose_bottle": "loose\nbottleneck",
    "tight_bottle": "tight\nbottleneck",
}

for param in ["beta", "shift"]:
    data = model_vs_truth_df[model_vs_truth_df["parameter"] == param].copy()
    data["measurement_type"] = data["measurement_type"].replace(bottlenecks)
    data["library"] = data["library"].replace({"lib_1": "rep1", "lib_2": "rep2"})

    fig, axes = plt.subplots(1, 3, figsize=[9, 3], sharey=True)
    for i, (mt, mt_df) in enumerate(data.groupby("measurement_type")):
        ax = axes[i]
        for lib, lib_df in mt_df.groupby("library"):
            ax.plot(lib_df["fusionreg"], lib_df["corr"], marker="o", label=lib)
        ax.set_title(mt)
        ax.set_xlabel("fusionreg (λ)")
        if i == 0:
            ax.set_ylabel(f"{param} correlation")
        ax.legend()
        sns.despine(ax=ax)
    plt.suptitle(f"Model vs Truth: {param}")
    plt.tight_layout()
    plt.show()

## Shift sparsity

In [ ]:
data = sparsity_df.copy()
data["measurement_type"] = data["measurement_type"].replace(bottlenecks)

fig, axes = plt.subplots(1, 3, figsize=[9, 3], sharey=True)
for i, (mt, mt_df) in enumerate(data.groupby("measurement_type")):
    ax = axes[i]
    for (lib, mtype), sub in mt_df.groupby(["library", "mut_type"]):
        ax.plot(sub["fusionreg"].astype(str), sub["sparsity"], marker="o", label=f"{lib} {mtype}")
    ax.set_title(mt)
    ax.set_xlabel("fusionreg (λ)")
    if i == 0:
        ax.set_ylabel("sparsity")
    ax.legend(fontsize=7)
    sns.despine(ax=ax)
plt.suptitle("Shift Sparsity")
plt.tight_layout()
plt.show()

## Cross-validation loss

In [ ]:
data = cv_loss_df.copy()
data["measurement_type"] = data["measurement_type"].replace(bottlenecks)

fig, axes = plt.subplots(1, 3, figsize=[9, 3], sharey=True)
for i, (mt, mt_df) in enumerate(data.groupby("measurement_type")):
    ax = axes[i]
    for (lib, ds), sub in mt_df.groupby(["library", "dataset"]):
        ax.plot(sub["fusionreg"].astype(str), sub["loss"], marker="o", label=f"{lib} {ds}")
    ax.set_title(mt)
    ax.set_xlabel("fusionreg (λ)")
    if i == 0:
        ax.set_ylabel("Huber loss")
    ax.legend(fontsize=7)
    sns.despine(ax=ax)
plt.suptitle("Cross-Validation Loss")
plt.tight_layout()
plt.show()

## Simulated mutation effect distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=[6, 2.7], sharex=True)

data_shifted = simulated_muteffects_df[simulated_muteffects_df["shift"] != 0]
sns.histplot(x="shift", data=data_shifted, bins=15, ax=axes[0])
axes[0].set(xlabel="shift", ylabel="number of mutations", title="shifted mutations")

sns.histplot(x="shift", data=simulated_muteffects_df, bins=15, ax=axes[1])
axes[1].set(xlabel="shift", ylabel="", title="all mutations")

plt.tight_layout()
sns.despine()
plt.show()

plt.figure(figsize=[3, 2])
sns.histplot(x="beta_h1", data=simulated_muteffects_df, bins=15, color="0.5")
plt.xlabel(r"mutational effect ($\beta_m$)")
plt.ylabel("number of mutations")
sns.despine()
plt.show()